In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# הגדרת רמת אופטימיזציה של ה-Transpiler

<details>
<summary><b>גרסאות חבילות</b></summary>

הקוד בדף זה פותח באמצעות הדרישות הבאות.
אנו ממליצים להשתמש בגרסאות אלו או בגרסאות חדשות יותר.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
מכשירים קוונטיים אמיתיים כפופים לרעש ושגיאות Gate, ולכן אופטימיזציה של המעגלים כדי להפחית את עומקם ומספר ה-Gate שלהם יכולה לשפר משמעותית את התוצאות המתקבלות מהרצת אותם מעגלים.
לפונקציה [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) יש ארגומנט מיקום חובה אחד, `optimization_level`, השולט בכמה מאמץ ה-Transpiler משקיע באופטימיזציה של מעגלים. ארגומנט זה יכול להיות מספר שלם המקבל אחד מהערכים 0, 1, 2 או 3.
רמות אופטימיזציה גבוהות יותר מייצרות מעגלים מאופטמים יותר על חשבון זמני קומפילציה ארוכים יותר.
הטבלה הבאה מסבירה את האופטימיזציות המבוצעות עם כל הגדרה.

<table>
  <thead>
    <tr>
      <th>רמת אופטימיזציה</th>
      <th>תיאור</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>0</td>
      <td>
        ללא אופטימיזציה: בדרך כלל משמש לאפיון חומרה
        - תרגום בסיסי
        - Layout/Routing: `TrivialLayout`, שבו נבחרים אותם מספרי Qubit פיזיים כמספרים וירטואליים ומוכנסים SWAPs כדי לגרום לזה לעבוד (באמצעות `SabreSwap`)
      </td>
    </tr>
    <tr>
      <td>1</td>
      <td>
        אופטימיזציה קלה:
        -   Layout/Routing: Layout נסיון ראשון עם `TrivialLayout`. אם נדרשים SWAPs נוספים, נמצא layout עם מספר מינימלי של SWAPs באמצעות `SabreSwap`, ואז משתמש ב-`VF2LayoutPostLayout` כדי לנסות לבחור את ה-Qubit הטובים ביותר בגרף.
        -   `InverseCancellation`
        -   אופטימיזציית Gate חד-קיוביטית
      </td>
    </tr>
    <tr>
      <td>2</td>
      <td>
        אופטימיזציה בינונית:
          - Layout/Routing: רמת אופטימיזציה 1 (ללא trivial) + הוריסטי מאופטמל עם עומק חיפוש גדול יותר וניסיונות של פונקציית אופטימיזציה. מכיוון ש-`TrivialLayout` אינו בשימוש, אין ניסיון להשתמש באותם מספרי Qubit פיזיים וירטואליים.
        -   `CommutativeCancellation`
      </td>
    </tr>
    <tr>
      <td>3</td>
      <td>
        אופטימיזציה גבוהה:
        - רמת אופטימיזציה 2 + הוריסטי מאופטמל על layout/routing עם מאמץ/ניסיונות נוספים
        - סינתזה מחדש של בלוקי Gate דו-קיוביטיים באמצעות [פירוק KAK של Cartan](https://arxiv.org/abs/quant-ph/0507171).
        - מעברים המפרים unitarity:
          * `OptimizeSwapBeforeMeasure`: מזיז את המדידות כדי להימנע מ-SWAPs
          * `RemoveDiagonalGatesBeforeMeasure`: מסיר Gateים לפני מדידות שלא ישפיעו על המדידות
      </td>
    </tr>
  </tbody>
</table>

## רמת אופטימיזציה בפעולה
מכיוון ש-Gateים דו-קיוביטיים הם בדרך כלל המקור המשמעותי ביותר לשגיאות, נוכל לכמת בקירוב את "יעילות החומרה" של ה-transpilation על ידי ספירת מספר ה-Gateים הדו-קיוביטיים במעגל המתקבל.
כאן, ננסה את רמות האופטימיזציה השונות על מעגל קלט המורכב מ-unitary אקראי אחריו Gate SWAP.

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import Operator, random_unitary

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qc = QuantumCircuit(2)
qc.append(rand_U, range(2))
qc.swap(0, 1)
qc.draw("mpl", style="iqp")

<Image src="../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/81173ebc-8359-48a6-b585-0477907b3b93-0.svg)

נשתמש ב-backend המדומה `FakeSherbrooke` בדוגמאות שלנו. ראשית, נבצע transpile באמצעות רמת אופטימיזציה 0.

In [2]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=0, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/40cdd173-b437-48b1-8928-741e8411342e-0.svg)

למעגל שעבר transpile יש שישה Gate ECR דו-קיוביטיים.

חזור על הפעולה עבור רמת אופטימיזציה 1:

In [3]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

backend = FakeSherbrooke()

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, seed_transpiler=12345
)
qc_t1_exact = pass_manager.run(qc)
qc_t1_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/2dab5def-a017-42e9-92d6-e043ac4065b2-0.svg)

למעגל שעבר transpile עדיין יש שישה Gate ECR, אך מספר ה-Gateים החד-קיוביטיים הצטמצם.

חזור על הפעולה עבור רמת אופטימיזציה 2:

In [4]:
pass_manager = generate_preset_pass_manager(
    optimization_level=2, backend=backend, seed_transpiler=12345
)
qc_t2_exact = pass_manager.run(qc)
qc_t2_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/77d76048-b1e8-4225-b35f-80dc9d458e8d-0.svg)

פעולה זו מניבה את אותן תוצאות כמו רמת אופטימיזציה 1. שים לב שהגדלת רמת האופטימיזציה לא תמיד עושה הבדל.

חזור שוב, עם רמת אופטימיזציה 3:

In [5]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend, seed_transpiler=12345
)
qc_t3_exact = pass_manager.run(qc)
qc_t3_exact.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/4109d0e2-df37-4850-8409-6b860c48595c-0.svg)

כעת, יש רק שלושה Gate ECR. אנו מקבלים תוצאה זו מכיוון שברמת אופטימיזציה 3, Qiskit מנסה לסנתז מחדש בלוקי Gate דו-קיוביטיים, וכל Gate דו-קיוביטי ניתן לממש באמצעות לכל היותר שלושה Gate ECR. נוכל לקבל אפילו פחות Gate ECR אם נגדיר את `approximation_degree` לערך קטן מ-1, ומאפשרים ל-Transpiler לבצע קירובים שעשויים להכניס שגיאה מסוימת בפירוק ה-Gate (ראה [פרמטרים נפוצים לשימוש ב-transpilation](common-parameters#approximation-degree)):

In [6]:
pass_manager = generate_preset_pass_manager(
    optimization_level=3,
    approximation_degree=0.99,
    backend=backend,
    seed_transpiler=12345,
)
qc_t3_approx = pass_manager.run(qc)
qc_t3_approx.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/set-optimization/extracted-outputs/bf239116-b8bb-42aa-a27a-89206d9e108a-0.svg)

למעגל זה יש רק שני Gate ECR, אך הוא מעגל משוער. כדי להבין כיצד ההשפעה שלו שונה מהמעגל המדויק, נוכל לחשב את ה-fidelity בין האופרטור האוניטרי שמעגל זה מממש, לבין האוניטרי המדויק. לפני ביצוע החישוב, נצמצם תחילה את המעגל שעבר transpile, המכיל 127 קיוביטים, למעגל המכיל רק את הקיוביטים הפעילים, שמספרם שניים.

In [7]:
import numpy as np


def trace_to_fidelity_2q(trace: float) -> float:
    return (4.0 + trace * trace.conjugate()) / 20.0


# Reduce circuits down to 2 qubits so they are easy to simulate
qc_t3_exact_small = QuantumCircuit.from_instructions(qc_t3_exact)
qc_t3_approx_small = QuantumCircuit.from_instructions(qc_t3_approx)

# Compute the fidelity
exact_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_exact_small).adjoint().data, UU))
)
approx_fid = trace_to_fidelity_2q(
    np.trace(np.dot(Operator(qc_t3_approx_small).adjoint().data, UU))
)
print(
    f"Synthesis fidelity\nExact: {exact_fid:.3f}\nApproximate: {approx_fid:.3f}"
)

Synthesis fidelity
Exact: 1.000+0.000j
Approximate: 0.992+0.000j


Adjusting the optimization level can change other aspects of the circuit too, not just the number of ECR gates. For examples of how setting optimization level changes the layout, see [Representing quantum computers](./represent-quantum-computers).

## Next steps

<Admonition type="tip" title="Recommendations">
    - To learn more about the `generate_preset_passmanager` function, start with the [Transpilation default settings and configuration options](defaults-and-configuration-options) topic.
    - Continue learning about transpilation with the [Transpiler stages](transpiler-stages) topic.
    - Try the [Compare transpiler settings](/docs/guides/circuit-transpilation-settings) guide.
    - Try the [Build repetition codes](/docs/tutorials/repetition-codes) tutorial.
    - See the [Transpile API documentation.](/docs/api/qiskit/transpiler)
</Admonition>